# Huấn luyện Persistent Homology trên toàn bộ dữ liệu Diginetica

## 1. Mục tiêu và giao thức đánh giá
Huấn luyện từ đầu mô hình gợi ý phiên với hàm mất mát CE + PH H₀/H₁. Đối chứng gồm CE và CE + cosine, cùng dữ liệu, seed và cấu hình tối ưu.

Toàn bộ tập train dùng để học; validation chọn checkpoint theo MRR@20; test chỉ đánh giá sau khi chọn checkpoint. Mặc định chạy GRU4Rec với hai seed. Có thể chọn NARM hoặc SASRec ở phần 4.

**Cấu hình chạy:** GPU T4 ×2, `RUN_FULL_TRAINING = True`. PH chạy trước trong mỗi seed, sau đó chạy các đối chứng. Save Version chạy toàn bộ notebook và lưu kết quả trong Output.

## 2. Chuẩn bị mã nguồn
Khôi phục mã nguồn và cấu hình đi kèm notebook.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, sys, zlib
ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/ph_full_data')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    bundle = json.loads(zlib.decompress(base64.b64decode('...')))
    for relative, content in bundle.items():
        destination = PROJECT_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(content, encoding='utf-8')
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p/'BTL-K03-tong-quan-de-tai.md').exists())
sys.path.insert(0, str(PROJECT_ROOT/'src'))
os.environ.update(PYTHONIOENCODING='utf-8', PYTHONUTF8='1', PYTHONDONTWRITEBYTECODE='1')
print('Project:', PROJECT_ROOT)

## 3. Kiểm tra dữ liệu
Xác thực bốn tệp dữ liệu bằng SHA-256 và thống kê số mẫu của từng tập.

In [ ]:
import pandas as pd
EXPECTED_HASHES = {'item_mapping.csv': '7ef8636a4e74c9471f0e2d9805ae5b5c3fe3186dcb5e094f6fd2a86d75097f4a', 'train_examples.parquet': 'eb41c516b26ea163ce6d9cceefd6b6817d6c897f49dcc23d4ab0136d69225d04', 'validation_examples.parquet': '52cc7af596b5b708162781b9f24267bf49c4a0fd5e43f3d5c8df9302639e194c', 'test_examples.parquet': '54b5014dbfbb8a43507b06655421bca92930e8563aad41295cb6deaa37260668'}
DATA = PROJECT_ROOT/'data/processed/diginetica'
if ON_KAGGLE:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('item_mapping.csv')]
    source = next((p for p in candidates if all((p/name).exists() and
        hashlib.sha256((p/name).read_bytes()).hexdigest() == digest
        for name, digest in EXPECTED_HASHES.items())), None)
    if source is None:
        raise RuntimeError('Attach ngytfd/btl-k03-sasrec-data; expected data hashes not found.')
    DATA.mkdir(parents=True, exist_ok=True)
    for name in EXPECTED_HASHES:
        shutil.copy2(source/name, DATA/name)
assert all(hashlib.sha256((DATA/name).read_bytes()).hexdigest() == digest
           for name, digest in EXPECTED_HASHES.items())
SPLIT_COUNTS = {split: len(pd.read_parquet(DATA/f'{split}_examples.parquet'))
                for split in ['train', 'validation', 'test']}
display(pd.DataFrame([{'split': k, 'examples': v} for k,v in SPLIT_COUNTS.items()]))
assert SPLIT_COUNTS == {'train': 498849, 'validation': 29828, 'test': 40001}

## 4. Cấu hình huấn luyện
Chọn kiến trúc và bật huấn luyện đầy đủ. Mỗi seed dùng một GPU; ba phương pháp lần lượt chạy trong từng seed.

PH so khớp persistence diagram của đám mây biểu diễn phiên với đám mây embedding target tương ứng trong cùng batch. Target được dừng gradient ở nhánh tham chiếu. Mỗi batch dùng tối đa 4 đám mây, mỗi đám mây có tối đa 6 đỉnh: anchor và 5 láng giềng có target khác nhau. Anchor có tần suất train từ 1 đến 19. Việc lấy đám mây không làm giảm số mẫu train dùng cho CE.

Hàm mất mát: **L = L_CE + 0,15 × (L_H₀ + L_H₁)**. Khoảng cách trên vector chuẩn hóa L2; Vietoris–Rips trên trường F₂; so khớp Wasserstein bình phương có ghép với đường chéo. Khoảng essential H₀ không đưa vào loss. H₁ có thể rỗng; khi đó không có thành phần H₁ cho đám mây đó.

Cosine là đối chứng căn chỉnh hướng phiên–target, không tính persistence diagram. Không dùng loss phân tán bổ sung.

In [ ]:
from datetime import datetime, timezone, timedelta
import copy, subprocess, time
import numpy as np
import torch
from btl_k03.utils import load_yaml, load_experiment_config

RUN_FULL_TRAINING = True
MODELS = ['gru4rec']  # Choose: ['gru4rec'], ['narm'], or ['sasrec'].
METHODS = ['ph_h0_h1', 'baseline', 'cosine']
SEEDS = [2024, 2025]
STUDY = load_yaml(PROJECT_ROOT/'configs/topology/ph_full_data.yaml')
assert MODELS and set(MODELS) <= set(STUDY['available_models'])
assert set(METHODS) == set(STUDY['methods'])
assert SEEDS == STUDY['seeds']
torch.set_num_threads(2)
GPU_COUNT = min(torch.cuda.device_count(), 2)
if RUN_FULL_TRAINING and GPU_COUNT < 1:
    raise RuntimeError('Enable a GPU before full training.')
RUN_ID = datetime.now(timezone(timedelta(hours=7))).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT = PROJECT_ROOT/'results/ph_full_data'/RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)
LOCKED = dict(STUDY, default_models=MODELS, methods=METHODS, seeds=SEEDS,
              data_sha256=EXPECTED_HASHES, split_counts=SPLIT_COUNTS)
(OUTPUT/'locked_study.json').write_text(json.dumps(LOCKED, indent=2), encoding='utf-8')
print('GPU count:', GPU_COUNT, '| Torch:', torch.__version__)
print('Full training:', RUN_FULL_TRAINING, '| Architectures:', MODELS)
print('Output:', OUTPUT)
display(pd.DataFrame([STUDY['training']]))

## 5. Kiểm tra phép tính PH
Đối chiếu H₁ của hình vuông đơn vị, khoảng cách giữa hai diagram giống nhau và gradient qua đầu mút persistence.

In [ ]:
from btl_k03.topology.persistent_homology import vietoris_rips_persistence, diagram_wasserstein_squared
square = torch.tensor([[0.,0.],[1.,0.],[1.,1.],[0.,1.]], requires_grad=True)
diagrams = vietoris_rips_persistence(square, max_dimension=1)
h1 = diagrams[1]
assert h1.shape == (1,2), h1
assert torch.allclose(h1.detach(), torch.tensor([[1., 2**.5]]), atol=1e-6), h1
same = diagram_wasserstein_squared(h1, h1.detach())
assert abs(float(same.detach())) < 1e-8
loss = (h1[:,1]-h1[:,0]).sum()
loss.backward()
assert torch.isfinite(square.grad).all() and square.grad.abs().sum() > 0
print('PASS: square H1 = [1, sqrt(2)], identical distance = 0, finite nonzero gradient.')

## 6. Thiết lập tiến trình huấn luyện
Mỗi tiến trình có seed và thiết bị riêng. Các đối chứng cùng seed khởi tạo từ cùng trọng số; checkpoint được chọn theo validation MRR@20.

In [ ]:
WORKER_SOURCE = "from pathlib import Path\nimport argparse, copy, gc, json, sys\nimport torch\n\ndef run_worker(root, output, model, seed, device, mode):\n    sys.path.insert(0, str(root/'src'))\n    from btl_k03.utils import load_yaml, load_experiment_config\n    from btl_k03.topology.topo_trainer import train_and_evaluate_topology_seed\n    study = json.loads((output/'locked_study.json').read_text(encoding='utf-8'))\n    # Path, not str: load_yaml uses Path.open.\n    recipe = load_yaml(root/'configs/topology/ph_full_data.yaml')\n    torch.set_num_threads(2)\n    full = mode == 'full'\n    for method in study['methods']:\n        config = load_experiment_config(model)\n        config['training'].update(copy.deepcopy(recipe['training']))\n        config['early_stopping'].update(copy.deepcopy(recipe['early_stopping']))\n        config['topology_params'] = copy.deepcopy(recipe['topology_params'])\n        if not full:\n            config['training'].update(epochs=1, train_batch_size=32, eval_batch_size=32)\n            config['topology_params'].update(max_clouds=2, neighbors=4)\n        target = output/('runs' if full else 'technical_checks')/model/f'seed_{seed}'/method\n        result = train_and_evaluate_topology_seed(seed, config=config,\n            processed_dir=root/'data/processed/diginetica', device_str=device,\n            model_name=model, method=method, output_dir=target,\n            train_limit=None if full else 64, validation_limit=None if full else 64,\n            init_checkpoint=None, evaluate_test=full)\n        assert result['frozen_embedding_max_abs_drift'] == 0\n        assert ('test' in result) == full\n        if full:\n            assert result['status'] == 'full_training'\n            assert result['experiment']['train_examples'] == study['split_counts']['train']\n            assert result['experiment']['validation_examples'] == study['split_counts']['validation']\n        print('FINISHED', model, seed, method, result['status'], flush=True)\n        gc.collect()\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n\nif __name__ == '__main__':\n    p = argparse.ArgumentParser()\n    for key in ['root','output','model','device','mode']: p.add_argument('--'+key, required=True)\n    p.add_argument('--seed', type=int, required=True)\n    args = p.parse_args()\n    run_worker(Path(args.root), Path(args.output), args.model, args.seed, args.device, args.mode)\n"
WORKER = OUTPUT/'train_worker.py'
WORKER.write_text(WORKER_SOURCE, encoding='utf-8')
def worker_command(model, seed, device, mode):
    return [sys.executable, '-u', '-X', 'utf8', str(WORKER), '--root', str(PROJECT_ROOT),
            '--output', str(OUTPUT), '--model', model, '--seed', str(seed),
            '--device', device, '--mode', mode]

## 7. Kiểm tra vận hành
Kiểm tra đọc cấu hình, train, chọn checkpoint và xuất kết quả cho cả ba kiến trúc. Mỗi lượt chỉ dùng 64 mẫu để kiểm tra mã; không dùng các lượt này để kết luận hiệu quả nghiên cứu.

In [ ]:
QA_DEVICE = 'cuda:0' if GPU_COUNT else 'cpu'
for architecture in STUDY['available_models']:
    subprocess.run(worker_command(architecture, 2024, QA_DEVICE, 'check'),
                   cwd=PROJECT_ROOT, check=True)
    base = OUTPUT/'technical_checks'/architecture/'seed_2024'
    snapshots = [json.loads((base/m/'metrics.json').read_text(encoding='utf-8')) for m in METHODS]
    assert len({r['experiment']['initial_embedding_sha256'] for r in snapshots}) == 1
(OUTPUT/'technical_checks_passed.json').write_text(json.dumps({
    'architectures': STUDY['available_models'], 'methods': METHODS,
    'scope': 'technical checks only; 64 train / 64 validation, no test'}, indent=2), encoding='utf-8')
print('PASS: all architectures and methods; matched initial embeddings; test untouched.')

## 8. Huấn luyện trên toàn bộ tập train
Huấn luyện tối đa 30 epoch, dừng sớm sau 5 epoch validation MRR@20 không cải thiện đủ 0,0001. Hai GPU chạy hai seed độc lập. Nhật ký được cập nhật mỗi 100 batch và mỗi epoch.

In [ ]:
if RUN_FULL_TRAINING:
    for architecture in MODELS:
        pending = list(SEEDS)
        running = []
        while pending or running:
            occupied = {slot for _,slot,_,_ in running}
            for slot in range(max(1,GPU_COUNT)):
                if pending and slot not in occupied:
                    seed = pending.pop(0)
                    log_path = OUTPUT/f'{architecture}_seed_{seed}.log'
                    handle = log_path.open('w', encoding='utf-8')
                    process = subprocess.Popen(worker_command(architecture, seed, f'cuda:{slot}', 'full'),
                        cwd=PROJECT_ROOT, stdout=handle, stderr=subprocess.STDOUT)
                    running.append((process,slot,seed,handle))
                    print('START', architecture, seed, f'cuda:{slot}', flush=True)
            time.sleep(10)
            for entry in list(running):
                process,slot,seed,handle = entry
                if process.poll() is not None:
                    handle.close()
                    running.remove(entry)
                    if process.returncode:
                        for other,_,_,fh in running:
                            other.terminate(); other.wait(); fh.close()
                        raise RuntimeError(f'Worker failed: {architecture}, seed {seed}. Read {architecture}_seed_{seed}.log')
                    print('DONE', architecture, seed, flush=True)
            for _,_,seed,_ in running:
                lines = (OUTPUT/f'{architecture}_seed_{seed}.log').read_text(encoding='utf-8').splitlines()
                if lines: print(f'{architecture}/{seed}: {lines[-1]}', flush=True)
else:
    print('Full training is OFF. Set RUN_FULL_TRAINING = True in section 4, then Run All.')

## 9. Tổng hợp kết quả và độ bất định
Xuất chỉ số test theo seed, trung bình hai seed và khoảng tin cậy bootstrap theo phiên cho PH so với CE/cosine. Báo cáo Recall, MRR, NDCG, coverage, Gini và ARP; không suy diễn cải thiện từ hình vẽ.

In [ ]:
import matplotlib.pyplot as plt
from btl_k03.statistics import clustered_delta
if RUN_FULL_TRAINING:
    records, comparisons = [], []
    for architecture in MODELS:
        for seed in SEEDS:
            base = OUTPUT/'runs'/architecture/f'seed_{seed}'
            for method in METHODS:
                result = json.loads((base/method/'metrics.json').read_text(encoding='utf-8'))
                row = {'model':architecture, 'seed':seed, 'method':method,
                       'best_epoch':result['best_epoch'], 'seconds':result['seconds']}
                for group in result['test']['ranking_by_group']:
                    for metric in ['recall@20','mrr@20','ndcg@20']:
                        row[group['target_group']+'_'+metric] = group[metric]
                row.update(result['test']['popularity_bias'])
                records.append(row)
            ph = pd.read_csv(base/'ph_h0_h1/test_per_query.csv.gz')
            for control in ['baseline','cosine']:
                ce = pd.read_csv(base/control/'test_per_query.csv.gz')
                for group in ['all_warm','rare']:
                    for metric in ['hit','reciprocal_rank','ndcg']:
                        delta = clustered_delta(ce, ph, metric, group, 2000,
                                                np.random.default_rng(seed))
                        comparisons.append(dict(model=architecture, seed=seed,
                            control=control, treatment='ph_h0_h1', group=group, metric=metric, **delta))
    summary = pd.DataFrame(records)
    summary.to_csv(OUTPUT/'test_results_by_seed.csv', index=False)
    means = summary.groupby(['model','method']).mean(numeric_only=True).drop(columns='seed').reset_index()
    means.to_csv(OUTPUT/'test_results_mean.csv', index=False)
    ci = pd.DataFrame(comparisons)
    ci.to_csv(OUTPUT/'test_paired_session_bootstrap.csv', index=False)
    display(summary)
    display(means)
    display(ci)
    for architecture in MODELS:
        subset = means[means.model.eq(architecture)].set_index('method').reindex(METHODS)
        fig, axes = plt.subplots(1,2,figsize=(12,4))
        for ax, metric in zip(axes,['all_warm_recall@20','rare_recall@20']):
            ax.bar(subset.index,100*subset[metric]); ax.set_ylabel(metric+' (%)'); ax.set_ylim(bottom=0)
        fig.tight_layout(); fig.savefig(OUTPUT/f'{architecture}_test_recall.png',dpi=180)
        plt.show(); plt.close(fig)
    print('CI is per seed, clustered by session; two seeds do not establish broad generalization.')
else:
    print('No full-training results yet.')

## 10. Phân tích cấu trúc embedding trước và sau huấn luyện
Với mỗi seed, chọn 3 item rare, 1 medium và 1 popular theo thống kê train. Cố định 12 ID láng giềng từ mô hình CE, rồi tính lại diagram/barcode trên cùng ID ở khởi tạo, sau CE và sau PH.

Hình này mô tả cấu trúc embedding của các item được chọn. Đây không phải biểu đồ loss session–target trong lúc train và không tự chứng minh chất lượng gợi ý tăng. Dữ liệu số, ID item và quy tắc chọn mẫu được lưu cùng hình.

In [ ]:
from btl_k03.topology.study_artifacts import export_fixed_cloud_comparison
if RUN_FULL_TRAINING:
    for architecture in MODELS:
        for seed in SEEDS:
            base = OUTPUT/'runs'/architecture/f'seed_{seed}'
            export_fixed_cloud_comparison(base/'ph_h0_h1', base/'baseline',
                OUTPUT/'geometry'/architecture/f'seed_{seed}', seed, **STUDY['geometry_export'])
    print('Saved fixed-ID diagrams, barcodes, raw arrays and selection provenance.')
else:
    print('Geometry figures will be generated after full training.')

## 11. Lưu tệp kết quả
Lưu cấu hình, nhật ký, checkpoint, lịch sử train, dự đoán test, bảng chỉ số và hình PH. Tải toàn bộ Output sau khi Kaggle hoàn thành phiên chạy.

In [ ]:
manifest = {'run_full_training': RUN_FULL_TRAINING, 'models':MODELS, 'methods':METHODS,
    'seeds':SEEDS, 'output':str(OUTPUT), 'data_sha256':EXPECTED_HASHES,
    'technical_checks_passed':(OUTPUT/'technical_checks_passed.json').exists(),
    'full_results_ready':(OUTPUT/'test_results_by_seed.csv').exists()}
(OUTPUT/'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))